# **Build a Mini-BERT Model From Scratch**

In [19]:
#Import Libraries

import torch
import torch.nn as nn 
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB
from transformers import BertTokenizer
import requests
from zipfile import ZipFile
from datasets import Dataset
import pandas as pd
import json
from torchtext.functional import pad_sequence
from torch.utils.data import DataLoader
from tensorflow import Tensor
import math

In [2]:
train_iter, val_iter = IMDB()


In [3]:
#define tokenizor
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [4]:
def download (url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(response.content)
    else:
        print(f"Failed to download. Status code: {response.status_code}")

In [5]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/bZaoQD52DcMpE7-kxwAG8A.zip'
filename = 'BERT-dataset'

In [6]:
download(url, filename)

In [7]:
def unzip_file (filename,directory):
    with ZipFile(filename, 'r') as unzip_f:
        unzip_f.extractall(directory)

In [8]:
directory = 'D:\Projects\mini-bert-nsp-mlm-pretraining'

In [9]:
unzip_file(filename,directory)

In [10]:
data = pd.read_csv('bert_dataset/bert_train_data.csv')
data.iloc[2]

Original Text    I rented I AM CURIOUS-YELLOW from my video sto...
BERT Input       [1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...
BERT Label       [0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...
Segment Label    1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...
Is Next                                                          1
Name: 2, dtype: object

### **Build Dataset Class**

In [11]:
class BertDatatsetCSV(Dataset):
    
    def __init__(self,filename):
        
        self.dataset = pd.read_csv(filename) 
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
     
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self,idx):
        row =  self.dataset.iloc[idx]
        
        try:
            
            bert_input = torch.tensor(json.loads(row['BERT input']), dtype=torch.long)
            bert_label = torch.tensor(json.loads(row['BERT label']), dtype=torch.long)
            segment_label = torch.tensor([int(x) for x in row['Segment Label'].split(',')], dtype=torch.long) 
            is_Next = torch.tensor(row['is Next'], dtype=torch.long)
            original_text = row['Original Text']
            
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON for row {idx}: {e}")
            print("BERT Input:", row['BERT Input'])
            print("BERT Label:", row['BERT Label'])
            # Handle the error, e.g., by skipping this row or using default values
            return None  # or some default values
    
        #Tokenize the original text
        #This creates:
            #input_ids → token numbers
            #attention_mask → tells model what is padding
        encoded_text = self.tokenizer.encode_plus(
            original_text,
            add_special_tokens = True,
            max_length = 512,
            padding = "max_length",
            truncation = True,
            return_tensor = 'pt'
        )
        
        #Why squeeze()?
            #return_tensors="pt" adds batch dimension → shape (1, 512)
            #.squeeze() → (512,)

        input_ids = encoded_text['input_ids'].squeeze()
        attention_mask = encoded_text['attention_mask'].squeeze()
    
        return (bert_input, bert_label, segment_label, is_Next,input_ids,attention_mask,original_text)
         

### **Build Collate Function**

In [12]:
PAD_IDX = 0

def collate_func(batch):
    bert_input_batch, bert_label_batch,segement_label_batch, is_next_batch, input_ids_batch, attention_mask_batch, original_text_batch = [],[],[],[],[],[],[]
    
    for bert_input, bert_label,segment_label, is_next, input_ids, attention_mask, original_text in batch:
        
        bert_input_batch.append(torch.tensor(bert_input, dtype=torch.long))
        bert_label_batch.append(torch.tensor(bert_label,dtype=torch.long))
        segement_label_batch.append(torch.tensor(segment_label,dtype=torch.long))
        is_next_batch.append(is_next)
        input_ids_batch.append(input_ids)
        attention_mask_batch.append(attention_mask)
        original_text_batch.append(original_text)
        
    bert_input_batch = pad_sequence(bert_input_batch,batch_first=True,padding_value=PAD_IDX)
    bert_label_batch = pad_sequence(bert_label_batch,batch_first=True,padding_value=PAD_IDX)
    segement_label_batch = pad_sequence(segement_label_batch,batch_first=True,padding_value=PAD_IDX)
    is_next_batch = torch.tensor(is_next_batch,dtype=torch.long)
    
    return bert_input_batch,bert_label_batch,segement_label_batch,is_next_batch

### **Initialize the Dataset and Dataloaders**

In [13]:
train_dataset = BertDatatsetCSV("./bert_dataset/bert_train_data.csv")
test_dataset = BertDatatsetCSV("./bert_dataset/bert_test_data.csv")

In [14]:
train_dataset.__len__()

170540

In [15]:
test_dataset.__len__()

167449

In [16]:
batch_size = 2

In [17]:
train_dataloader = DataLoader(train_dataset,batch_size=batch_size, collate_fn=collate_func,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size,shuffle=True, collate_fn=collate_func)

### **Model Implementation**

Model Creation (BERT Embeddings)

In BERT, three types of embeddings are combined to represent each input token:

1. Token Embedding

This is the basic representation of each word/token.

* Maps every token to a fixed-size dense vector
* Learns semantic meaning of words
* Captures relationships between tokens based on context

👉 Think of it as: “What does this word mean?”

2. Positional Embedding

Transformers process tokens in parallel, so they don’t naturally understand order.

* Adds position information to each token
* Encodes where a token appears in the sequence
* Helps the model understand word order and structure

👉 Think of it as: “Where is this word in the sentence?”

3. Segment Embedding

Used when BERT processes multiple sentences (e.g., sentence pairs).

* Assigns different embeddings to different segments (Sentence A, Sentence B)
* Helps distinguish between parts of the input
* Useful for tasks like question answering and next sentence prediction

👉 Think of it as: “Which sentence does this word belong to?”

Final Input Representation

The final embedding for each token is the sum of all three:

* Final Embedding = Token Embedding + Positional Embedding + Segment Embedding

This combined representation allows BERT to understand meaning, order, and context simultaneously.

#### **Token Embeddings**

In [20]:
class TokenEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, num_dims):
        super().__init__()
        self.num_dims = num_dims
        self.embeddings = nn.Embedding(vocab_size, num_dims)
    
    def forward(self, input_tokens:Tensor):
        embedding_out = self.embeddings(input_tokens.long())
        return embedding_out * math.sqrt(self.num_dims)